# BBO Capstone — Round 11 Query Generation
**Imperial Business School | Executive Master in ML/AI**  
**Candidate:** Gian Franco Cattaneo  
**Module:** 21.x — Refining Strategies for Black-Box Optimisation (Clustering Lens)  
**Date:** 2026-06-16  

---

## Pipeline Architecture
- **GP Kernel:** ConstantKernel (amplitude) x Matern-5/2 (ARD) + WhiteKernel (noise)
- **Scaler:** StandardScaler on inputs
- **Acquisition:** Expected Improvement (EI) — maximisation formulation
- **Optimiser:** L-BFGS-B with 25 restarts over 4,000 warm-start candidates
- **Objective:** all 8 functions treated as maximisation targets
- **Data:** 10 rounds x 8 functions = 80 evaluated points (Round 10 results ingested)

## Round 11 doctrine
By Round 11 the GP-EI acquisition is exploration-dominated on the converged functions, so it
is retained as a **cross-check** while **cluster geometry** (top-cluster centroid,
trust-region contraction, boundary tightening, damped gradient steps) is the decision rule.

In [1]:
# ============================================================
# CELL 1 — IMPORTS AND GLOBAL CONFIGURATION
# ============================================================
import numpy as np
import warnings
from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

N_RESTARTS   = 25
N_CANDIDATES = 4000
DOMAIN_LOWER = 0.0
DOMAIN_UPPER = 0.999999
N_FUNCTIONS  = 8
dims         = [2, 2, 3, 4, 4, 5, 6, 8]
print('Libraries loaded.')
print(f'  Restarts={N_RESTARTS}  Candidates={N_CANDIDATES}  Domain=[{DOMAIN_LOWER},{DOMAIN_UPPER}]')

Libraries loaded.
  Restarts=25  Candidates=4000  Domain=[0.0,0.999999]


In [2]:
# ============================================================
# CELL 2 — COMPLETE DATASET: ROUNDS 1-10  (index 0=f1 d=2 ... 7=f8 d=8; maximisation)
# ============================================================
all_inputs = [
    # --- Round 1 (W12 26/04) ---
    [np.array([0.034388, 0.909319]),
     np.array([0.695196, 0.395970]),
     np.array([0.548145, 0.174647, 0.303245]),
     np.array([0.440429, 0.425456, 0.378357, 0.397088]),
     np.array([0.000000, 0.675974, 0.999999, 0.999999]),
     np.array([0.464677, 0.242110, 0.574863, 0.999999, 0.000000]),
     np.array([0.000000, 0.241713, 0.327655, 0.218095, 0.375335, 0.747501]),
     np.array([0.064016, 0.008062, 0.123268, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 2 (W13 06/05) ---
    [np.array([0.999999, 0.999999]),
     np.array([0.698486, 0.000000]),
     np.array([0.850892, 0.035316, 0.936193]),
     np.array([0.999999, 0.000000, 0.000000, 0.365908]),
     np.array([0.000000, 0.000000, 0.999999, 0.999999]),
     np.array([0.142733, 0.321812, 0.416485, 0.999999, 0.304415]),
     np.array([0.000000, 0.302741, 0.000000, 0.187177, 0.000000, 0.167182]),
     np.array([0.096074, 0.000000, 0.581701, 0.000000, 0.999999, 0.383890, 0.202189, 0.999999])],

    # --- Round 3 (W14 08/05) ---
    [np.array([0.250000, 0.250000]),
     np.array([0.695000, 0.396000]),
     np.array([0.300000, 0.500000, 0.700000]),
     np.array([0.440000, 0.425000, 0.378000, 0.397000]),
     np.array([0.000000, 0.850000, 0.999999, 0.999999]),
     np.array([0.500000, 0.500000, 0.500000, 0.500000, 0.500000]),
     np.array([0.000000, 0.242000, 0.328000, 0.218000, 0.375000, 0.748000]),
     np.array([0.064000, 0.008000, 0.120000, 0.000000, 0.999999, 0.382000, 0.031000, 0.806000])],

    # --- Round 4 (W15 13/05) ---
    [np.array([0.500000, 0.500000]),
     np.array([0.700000, 0.200000]),
     np.array([0.950000, 0.010000, 0.990000]),
     np.array([0.999999, 0.000000, 0.000000, 0.700000]),
     np.array([0.000000, 0.000000, 0.500000, 0.500000]),
     np.array([0.300000, 0.400000, 0.600000, 0.200000, 0.600000]),
     np.array([0.000000, 0.150000, 0.000000, 0.100000, 0.000000, 0.100000]),
     np.array([0.100000, 0.000000, 0.800000, 0.000000, 0.999999, 0.380000, 0.350000, 0.999999])],

    # --- Round 5 (W16 18/05) ---
    [np.array([0.472781, 0.505546]),
     np.array([0.695211, 0.395970]),
     np.array([0.511275, 0.215264, 0.371049]),
     np.array([0.455000, 0.415000, 0.385000, 0.395000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.758817, 0.272673, 0.522143, 0.999999, 0.000000]),
     np.array([0.000000, 0.260000, 0.340000, 0.232000, 0.395000, 0.752000]),
     np.array([0.040000, 0.000000, 0.090000, 0.005000, 0.999999, 0.367013, 0.020000, 0.780000])],

    # --- Round 6 (W17 22/05) ---
    [np.array([0.445562, 0.511092]),
     np.array([0.693000, 0.397000]),
     np.array([0.490000, 0.230000, 0.395000]),
     np.array([0.430000, 0.430000, 0.375000, 0.400000]),
     np.array([0.005000, 0.999999, 0.999999, 0.999999]),
     np.array([0.450000, 0.240000, 0.580000, 0.999999, 0.000000]),
     np.array([0.000000, 0.238000, 0.325000, 0.215000, 0.370000, 0.743000]),
     np.array([0.063000, 0.008000, 0.123000, 0.000000, 0.999999, 0.382000, 0.031000, 0.807000])],

    # --- Round 7 (W18 27/05) ---
    [np.array([0.475000, 0.503000]),
     np.array([0.697000, 0.393000]),
     np.array([0.478000, 0.223000, 0.408000]),
     np.array([0.420000, 0.440000, 0.373000, 0.403000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.468000, 0.241000, 0.572000, 0.999999, 0.000000]),
     np.array([0.000000, 0.235000, 0.322000, 0.212000, 0.367000, 0.740000]),
     np.array([0.064016, 0.008062, 0.124000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 8 (W19 30/05) ---
    [np.array([0.477000, 0.501000]),
     np.array([0.695000, 0.394000]),
     np.array([0.465000, 0.222000, 0.421000]),
     np.array([0.428000, 0.432000, 0.374000, 0.401000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.460000, 0.242000, 0.575000, 0.999999, 0.000000]),
     np.array([0.000000, 0.232000, 0.319000, 0.209000, 0.364000, 0.737000]),
     np.array([0.064016, 0.008062, 0.126000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 9 (W20 03/06) ---
    [np.array([0.479000, 0.499000]),
     np.array([0.695200, 0.396500]),
     np.array([0.480000, 0.221000, 0.406000]),
     np.array([0.426000, 0.434000, 0.373000, 0.402000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.465000, 0.241000, 0.576000, 0.999999, 0.000000]),
     np.array([0.000000, 0.229000, 0.316000, 0.206000, 0.361000, 0.734000]),
     np.array([0.064016, 0.008062, 0.128000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 10 (W21 10/06) ---
    [np.array([0.481000, 0.497000]),
     np.array([0.696000, 0.395500]),
     np.array([0.476000, 0.225000, 0.410000]),
     np.array([0.430000, 0.430000, 0.376000, 0.399000]),
     np.array([0.003000, 0.999999, 0.999999, 0.999999]),
     np.array([0.466000, 0.242000, 0.575000, 0.999999, 0.000000]),
     np.array([0.000000, 0.226000, 0.310000, 0.200000, 0.355000, 0.730000]),
     np.array([0.064016, 0.008062, 0.130000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],
]

all_outputs = [
    [-2.467474707e-270, 0.7237404633, -0.08911956876, 0.259575752, 2105.928152, -0.5507747203, 2.207308607, 9.85954861],  # R1
    [1.51764873e-192, 0.5297658866, -0.239824301, -27.85976797, 1616.625747, -1.004515324, 0.05097822865, 9.293376957],  # R2
    [9.79774841e-42, 0.5263661301, -0.113960203, 0.274808002, 2932.694991, -1.015926849, 2.207174611, 9.8591546],  # R3
    [2.675287991e-09, 0.5813540452, -0.459406581, -30.89444083, 83.9625, -1.223884841, 0.02363347322, 8.51290028],  # R4
    [8.168635328e-08, 0.6238852457, -0.07070838209, -0.3996600231, 4440.480873, -0.910578472, 2.113257173, 9.838749658],  # R5
    [-5.316626717e-07, 0.3979411318, -0.0529490459, 0.4636173327, 4440.48296, -0.5765502837, 2.237774337, 9.8591203],  # R6
    [1.311073283e-07, 0.487212023, -0.03529824726, 0.3634822934, 4440.480873, -0.6360717448, 2.250193136, 9.85957657],  # R7
    [1.74738013e-07, 0.5737744969, -0.04183377359, 0.4710590923, 4440.480873, -0.562003489, 2.261159335, 9.85963657],  # R8
    [2.216253425e-07, 0.3666514255, -0.04723162937, 0.4679683637, 4440.480873, -0.6061839383, 2.27064725, 9.85967257],  # R9
    [2.68334119e-07, 0.5861001087, -0.04317830246, 0.4507794885, 4440.482053, -0.5526416388, 2.276410698, 9.85968457],  # R10
]

N_ROUNDS = len(all_inputs)
assert N_ROUNDS == len(all_outputs) == 10
func_X = [np.array([all_inputs[r][f]  for r in range(N_ROUNDS)]) for f in range(N_FUNCTIONS)]
func_y = [np.array([all_outputs[r][f] for r in range(N_ROUNDS)]) for f in range(N_FUNCTIONS)]
print(f'Dataset: {N_ROUNDS} rounds x {N_FUNCTIONS} functions = {N_ROUNDS*N_FUNCTIONS} points')
for f in range(N_FUNCTIONS):
    b = int(np.argmax(func_y[f]))
    print(f'  f{f+1} (d={dims[f]}): best=R{b+1}  y*={func_y[f][b]:.8f}')

Dataset: 10 rounds x 8 functions = 80 points
  f1 (d=2): best=R10  y*=0.00000027
  f2 (d=2): best=R1  y*=0.72374046
  f3 (d=3): best=R7  y*=-0.03529825
  f4 (d=4): best=R8  y*=0.47105909
  f5 (d=4): best=R6  y*=4440.48296000
  f6 (d=5): best=R1  y*=-0.55077472
  f7 (d=6): best=R10  y*=2.27641070
  f8 (d=8): best=R10  y*=9.85968457


In [3]:
# ============================================================
# CELL 3 — GP-BO PIPELINE
# ============================================================
def build_kernel(d):
    return (ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-3, 10.0), nu=2.5)
            + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1)))

def fit_gp(X_raw, y):
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)
    gp = GaussianProcessRegressor(kernel=build_kernel(X_raw.shape[1]),
                                  n_restarts_optimizer=10, normalize_y=True, random_state=42)
    gp.fit(X_sc, y)
    return gp, scaler

def expected_improvement(x, gp, scaler, y_best, xi=0.01):
    x_sc      = scaler.transform(x.reshape(1, -1))
    mu, sigma = gp.predict(x_sc, return_std=True)
    mu, sigma = float(mu[0]), float(sigma[0])
    if sigma < 1e-12:
        return 0.0
    z = (mu - y_best - xi) / sigma
    return -((mu - y_best - xi) * norm.cdf(z) + sigma * norm.pdf(z))

def optimise_acquisition(gp, scaler, y_best, d):
    cands = np.random.uniform(DOMAIN_LOWER, DOMAIN_UPPER, (N_CANDIDATES, d))
    best_x, best_v = None, np.inf
    for c in cands:
        v = expected_improvement(c, gp, scaler, y_best)
        if v < best_v:
            best_v, best_x = v, c
    for _ in range(N_RESTARTS):
        x0  = np.random.uniform(DOMAIN_LOWER, DOMAIN_UPPER, d)
        res = minimize(expected_improvement, x0, args=(gp, scaler, y_best),
                       method='L-BFGS-B', bounds=[(DOMAIN_LOWER, DOMAIN_UPPER)] * d)
        if res.fun < best_v:
            best_v, best_x = res.fun, res.x
    return best_x, -best_v
print('Pipeline defined.')

Pipeline defined.


In [4]:
# ============================================================
# CELL 4 — GP-EI CROSS-CHECK (exploration reference, not the decision)
# ============================================================
print(f'{"Func":>5}  {"d":>2}  {"y_best":>12}  {"GP-EI x*":>50}  {"EI":>10}')
print('=' * 90)
gp_suggestions = []
for fi in range(N_FUNCTIONS):
    gp, scaler = fit_gp(func_X[fi], func_y[fi])
    y_best     = float(np.max(func_y[fi]))
    x_next, ei = optimise_acquisition(gp, scaler, y_best, dims[fi])
    gp_suggestions.append(x_next)
    xs = '  '.join(f'{v:.4f}' for v in x_next)
    print(f'  f{fi+1:<2}  {dims[fi]:>2}  {y_best:>12.6f}  [{xs}]  {ei:>10.3e}')
print('\nNote: on converged functions GP-EI proposes far-field corners (exploration')
print('term dominates at n=10) -> used only as a cross-check, not the decision.')

 Func   d        y_best                                            GP-EI x*          EI


  f1    2      0.000000  [0.3745  0.9507]   0.000e+00


  f2    2      0.723740  [0.1273  0.1757]   8.002e-04


  f3    3     -0.035298  [0.0000  0.0000  0.0000]   3.212e-02


  f4    4      0.471059  [0.4223  0.0000  0.0000  1.0000]   2.798e+00


  f5    4   4440.482960  [0.0286  0.9867  0.2460  0.4404]   2.676e+02


  f6    5     -0.550775  [0.2037  0.2368  0.6383  0.9987  0.5236]   3.112e-02


  f7    6      2.276411  [0.0043  0.0378  0.7966  0.1991  0.5200  0.3704]   2.050e-01


  f8    8      9.859685  [0.0268  0.3178  0.4552  0.3581  0.5013  0.6123  0.1181  0.9079]   9.712e-02

Note: on converged functions GP-EI proposes far-field corners (exploration
term dominates at n=10) -> used only as a cross-check, not the decision.


In [5]:
# ============================================================
# CELL 5 — CLUSTER GEOMETRY (the Round 11 decision basis)
# ============================================================
print('CLUSTER GEOMETRY — top-cluster centroid, last step, deceleration')
print('=' * 78)
for fi in range(N_FUNCTIONS):
    X, y = func_X[fi], func_y[fi]
    k    = min(4, len(y)); idx = np.argsort(y)[-k:]
    cen  = X[idx].mean(0)
    d2   = (y[-1] - y[-2]) - (y[-2] - y[-3])
    print(f'\nf{fi+1} (d={dims[fi]}):')
    print(f'  top-{k} centroid : {np.array2string(np.round(cen,4), separator=", ")}')
    print(f'  last 3 y        : {y[-3]:.6e} -> {y[-2]:.6e} -> {y[-1]:.6e}')
    print(f'  R9->R10 step    : {np.array2string(np.round(X[-1]-X[-2],4), separator=", ")}')
    print(f'  2nd diff        : {d2:+.3e}  ->  {"decelerating" if d2 < 0 else "steady/accel"}')

CLUSTER GEOMETRY — top-cluster centroid, last step, deceleration

f1 (d=2):
  top-4 centroid : [0.478, 0.5  ]
  last 3 y        : 1.747380e-07 -> 2.216253e-07 -> 2.683341e-07
  R9->R10 step    : [ 0.002, -0.002]
  2nd diff        : -1.786e-10  ->  decelerating

f2 (d=2):
  top-4 centroid : [0.6966, 0.3469]
  last 3 y        : 5.737745e-01 -> 3.666514e-01 -> 5.861001e-01
  R9->R10 step    : [ 0.0008, -0.001 ]
  2nd diff        : +4.266e-01  ->  steady/accel

f3 (d=3):
  top-4 centroid : [0.4748, 0.2228, 0.4112]
  last 3 y        : -4.183377e-02 -> -4.723163e-02 -> -4.317830e-02
  R9->R10 step    : [-0.004,  0.004,  0.004]
  2nd diff        : +9.451e-03  ->  steady/accel

f4 (d=4):
  top-4 centroid : [0.4285, 0.4315, 0.3745, 0.4005]
  last 3 y        : 4.710591e-01 -> 4.679684e-01 -> 4.507795e-01
  R9->R10 step    : [ 0.004, -0.004,  0.003, -0.003]
  2nd diff        : -1.410e-02  ->  decelerating

f5 (d=4):
  top-4 centroid : [0.002, 1.   , 1.   , 1.   ]
  last 3 y        : 4.440481e+03 

In [6]:
# ============================================================
# CELL 6 — ROUND 11 QUERIES (cluster-informed)
# ============================================================
round11_queries = [
    np.array([0.483000, 0.495000]),                                   # f1 ridge-follow x1+/x2-
    np.array([0.695400, 0.395600]),                                   # f2 centroid contraction (noisy peak)
    np.array([0.477500, 0.223500, 0.408500]),                         # f3 trust-region -> R7
    np.array([0.428500, 0.431500, 0.374500, 0.400500]),               # f4 plateau centroid
    np.array([0.010000, 0.999999, 0.999999, 0.999999]),               # f5 line search x1 -> 0.010
    np.array([0.465000, 0.242100, 0.574900, 0.999999, 0.000000]),     # f6 boundary-anchored contraction
    np.array([0.000000, 0.223000, 0.307000, 0.197000, 0.352000, 0.727000]),  # f7 damped -0.003 step
    np.array([0.064016, 0.008062, 0.133000, 0.000000,
              0.999999, 0.381742, 0.031402, 0.806010]),               # f8 bracket x3 -> 0.133
]
for fi, x in enumerate(round11_queries):
    assert len(x) == dims[fi], f'f{fi+1}: dimension mismatch'
    assert np.all((x >= DOMAIN_LOWER) & (x <= DOMAIN_UPPER)), f'f{fi+1}: out of bounds'
print('All dimension and bound checks passed.')

All dimension and bound checks passed.


In [7]:
# ============================================================
# CELL 7 — GP POSTERIOR DIAGNOSTICS AT ROUND 11 QUERIES
# ============================================================
mode = ['Ridge-follow x1+/x2-','Centroid contraction','Trust-region -> R7','Plateau centroid',
        'Line search x1->0.010','Boundary-anchored contraction','Damped -0.003 step','Bracket x3->0.133']
print(f'{"Func":>5}  {"y_best":>13}  {"mu(pred)":>12}  {"sigma":>10}  {"EI":>10}  Mode')
print('-' * 90)
for fi in range(N_FUNCTIONS):
    gp, scaler = fit_gp(func_X[fi], func_y[fi]); y_best = float(np.max(func_y[fi]))
    xq = round11_queries[fi]
    mu, sg = gp.predict(scaler.transform(xq.reshape(1,-1)), return_std=True)
    ei = -expected_improvement(xq, gp, scaler, y_best)
    print(f'  f{fi+1:<2}  {y_best:>13.6f}  {float(mu[0]):>12.6f}  {float(sg[0]):>10.4f}  {ei:>10.3e}  {mode[fi]}')

 Func         y_best      mu(pred)       sigma          EI  Mode
------------------------------------------------------------------------------------------


  f1        0.000000      0.000000      0.0000   0.000e+00  Ridge-follow x1+/x2-
  f2        0.723740      0.539679      0.0966   8.002e-04  Centroid contraction


  f3       -0.035298     -0.043628      0.0056   7.775e-07  Trust-region -> R7


  f4        0.471059      0.467176      0.0014   6.755e-27  Plateau centroid
  f5     4440.482960   4386.667313    274.7586   8.480e+01  Line search x1->0.010


  f6       -0.550775     -0.550213      0.0018   4.277e-11  Boundary-anchored contraction


  f7        2.276411      2.281850      0.0018   2.819e-06  Damped -0.003 step
  f8        9.859685      9.859761      0.0001   0.000e+00  Bracket x3->0.133


In [8]:
# ============================================================
# CELL 8 — ROUND 11 SUBMISSION STRINGS (PORTAL FORMAT x1-x2-...-xn)
# ============================================================
labels = ['F1 (d=2)','F2 (d=2)','F3 (d=3)','F4 (d=4)','F5 (d=4)','F6 (d=5)','F7 (d=6)','F8 (d=8)']
print('ROUND 11 — FINAL SUBMISSION STRINGS'); print('=' * 60)
submission = []
for fi in range(N_FUNCTIONS):
    s = '-'.join(f'{v:.6f}' for v in round11_queries[fi]); submission.append(s)
    print(f'{labels[fi]}:  {s}')
print('\n--- COPY-PASTE BLOCK ---')
for s in submission: print(s)

ROUND 11 — FINAL SUBMISSION STRINGS
F1 (d=2):  0.483000-0.495000
F2 (d=2):  0.695400-0.395600
F3 (d=3):  0.477500-0.223500-0.408500
F4 (d=4):  0.428500-0.431500-0.374500-0.400500
F5 (d=4):  0.010000-0.999999-0.999999-0.999999
F6 (d=5):  0.465000-0.242100-0.574900-0.999999-0.000000
F7 (d=6):  0.000000-0.223000-0.307000-0.197000-0.352000-0.727000
F8 (d=8):  0.064016-0.008062-0.133000-0.000000-0.999999-0.381742-0.031402-0.806010

--- COPY-PASTE BLOCK ---
0.483000-0.495000
0.695400-0.395600
0.477500-0.223500-0.408500
0.428500-0.431500-0.374500-0.400500
0.010000-0.999999-0.999999-0.999999
0.465000-0.242100-0.574900-0.999999-0.000000
0.000000-0.223000-0.307000-0.197000-0.352000-0.727000
0.064016-0.008062-0.133000-0.000000-0.999999-0.381742-0.031402-0.806010


In [9]:
# ============================================================
# CELL 9 — CUMULATIVE BEST TRACKER (ALL 10 ROUNDS)
# ============================================================
print('Cumulative best by round'); print('=' * 110)
print(f'{"Round":>6}' + ''.join(f'  {"f"+str(i+1):>11}' for i in range(N_FUNCTIONS))); print('-' * 110)
best = [-np.inf]*N_FUNCTIONS
for r in range(N_ROUNDS):
    row = f'  R{r+1:>3}'
    for fi in range(N_FUNCTIONS):
        best[fi] = max(best[fi], all_outputs[r][fi]); row += f'  {best[fi]:>11.4f}'
    print(row)
print('\nRegime summary entering Round 11:')
print('  Climbing chains : f1, f5, f7, f8   (continue / damp by deceleration)')
print('  Tight blobs     : f3, f4, f6       (contract to centroid / incumbent)')
print('  Noisy cluster   : f2               (re-test centroid of high-value sub-cluster)')

Cumulative best by round
 Round           f1           f2           f3           f4           f5           f6           f7           f8
--------------------------------------------------------------------------------------------------------------
  R  1      -0.0000       0.7237      -0.0891       0.2596    2105.9282      -0.5508       2.2073       9.8595
  R  2       0.0000       0.7237      -0.0891       0.2596    2105.9282      -0.5508       2.2073       9.8595
  R  3       0.0000       0.7237      -0.0891       0.2748    2932.6950      -0.5508       2.2073       9.8595
  R  4       0.0000       0.7237      -0.0891       0.2748    2932.6950      -0.5508       2.2073       9.8595
  R  5       0.0000       0.7237      -0.0707       0.2748    4440.4809      -0.5508       2.2073       9.8595
  R  6       0.0000       0.7237      -0.0529       0.4636    4440.4830      -0.5508       2.2378       9.8595
  R  7       0.0000       0.7237      -0.0353       0.4636    4440.4830      -0.5508   

## Round 11 Reflection — the search space as clusters

Three cluster behaviours now coexist, each demanding a different cue:

1. **Open chains still climbing** (f1, f5, f7, f8) — the cluster is a *directed path*; the cue is the displacement vector between successive centroids, with step length set by the second difference of `y` (continue where steady, damp where decelerating).
2. **Tight blobs around an incumbent best** (f3, f4, f6) — the cluster has stopped moving and outward probes regressed; the cue is intra-cluster distance, so contract toward the centroid/incumbent.
3. **Noise-dominated cluster** (f2) — near-identical inputs return a wide spread, so the signal is the centroid of the high-value sub-cluster, not any single observation.

GP-EI (Cell 4) is kept for transparency on exploration pressure, but the falsifiable cluster claims in Cell 5 drive the decision.